In [18]:
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import re
import shutil
import string
from sklearn.metrics import classification_report
import tensorflow as tf
from keras.models import Model
from tensorflow.keras import layers
from tensorflow.keras import losses
from tensorflow.keras.optimizers import Adam
from keras.callbacks import EarlyStopping,ModelCheckpoint
from tensorflow.keras.layers import Dense, Input, Dropout, Bidirectional, LSTM, Embedding, BatchNormalization,  Reshape, Conv2D, MaxPool2D, concatenate, Flatten, Activation
import torch
import numpy as np
from transformers import BertTokenizer, BertModel,RobertaTokenizer, RobertaModel,AutoTokenizer, AutoModel
import ast

In [19]:
tf.random.set_seed(2023)

In [20]:
# Verificar si la GPU está disponible
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [21]:
archivo_3 = '/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/ResultsGraph/DataProgramsandDescriptions-CatRetroalimentacion5000.xlsx'
train_full = pd.read_excel(archivo_3)

In [22]:
class WhileLoopFinder(ast.NodeVisitor):
    def __init__(self, source_code):
        self.source_code = source_code.splitlines()
        self.functions_with_while = []
        self.current_function = None

    def visit_FunctionDef(self, node):
        original_current_function = self.current_function
        self.current_function = {
            "function_name": node.name,
            "has_while_loop": False,
            "pre_while_code_lines": [],
            "while_loops": []
        }

        function_start_line = node.lineno - 1
        function_end_line = (node.end_lineno if hasattr(node, 'end_lineno') else len(self.source_code))
        function_lines = self.source_code[function_start_line:function_end_line]

        found_while = False
        for stmt in node.body:
            if isinstance(stmt, ast.While):
                self.current_function["has_while_loop"] = True
                found_while = True
                try:
                    while_condition = ast.unparse(stmt.test).strip()
                    print("OK condition")
                except AttributeError:
                    print("Error al obtener la condición del while")
                try:
                    while_body_code = ast.unparse(ast.Module(body=stmt.body, type_ignores=[])).strip()
                    print("Ok body")
                except AttributeError:
                    print("Error al obtener el código del cuerpo del while")

                self.current_function["while_loops"].append({
                    "condition": while_condition,
                    "body_code": while_body_code
                })
            elif not found_while:
                start_line = stmt.lineno - 1
                end_line = (stmt.end_lineno if hasattr(stmt, 'end_lineno') else stmt.lineno)
                self.current_function["pre_while_code_lines"].extend(self.source_code[start_line:end_line])
            self.generic_visit(stmt)

        if self.current_function["has_while_loop"]:
            last_while_end_line = None
            if self.current_function["while_loops"]:
                last_while_node = next((node for node in reversed(node.body) if isinstance(node, ast.While)), None)
                if last_while_node and hasattr(last_while_node, 'end_lineno'):
                    last_while_end_line = last_while_node.end_lineno
            post_while_code_lines = []
            if last_while_end_line:
                function_indent = len(self.source_code[node.lineno - 1]) - len(self.source_code[node.lineno - 1].lstrip())
                for line in self.source_code[last_while_end_line:function_end_line]:
                    if line.startswith(self.source_code[node.lineno - 1][:function_indent] + "    "):
                        post_while_code_lines.append(line[function_indent + 4:])
                    else:
                        post_while_code_lines.append(line[function_indent:])
            self.current_function["post_while_code_lines"] = "\n".join(post_while_code_lines).strip()
            self.current_function["pre_while_code_lines"] = "\n".join(self.current_function["pre_while_code_lines"]).strip()
            self.functions_with_while.append(self.current_function)

        self.current_function = original_current_function



In [23]:
def DGries_states(solution):
  try:
    # Crear el AST
    tree = ast.parse(solution)
    # Recorrer el AST con nuestro visitante
    finder = WhileLoopFinder(solution)
    finder.visit(tree)
    for func_info in finder.functions_with_while:
      initial_state=func_info["pre_while_code_lines"] if func_info["pre_while_code_lines"] else False
      end_state=""
      transformation_state=""
      for j, wl in enumerate(func_info["while_loops"]):
          end_state=wl["condition"]
          transformation_state=wl["body_code"]

      if not initial_state:
        initial_state=transformation_state
      return initial_state,transformation_state,end_state
  except Exception as error:
    return "Exception:"+str(error),"Exception:"+str(error),"Exception:"+str(error)




In [24]:
class Encoder:
  def __init__(self):
    self.is_loadtokenizers=False



  def tokenize_and_generate_embeddings_descriptions(self,description):
    if not self.is_loadtokenizers:
      raise ValueError("tokenizers dont load")
    # Tokenize input description and move tokens to GPU
    tokens = self.beart_tokenizer.encode_plus(description, return_tensors="pt", truncation=True)
    tokens = {key: value.to(device) for key, value in tokens.items()}
    # Generate embeddings
    with torch.no_grad():
        outputs = self.beart_model(**tokens)
    # Extract embeddings for all tokens
    desc_embeddings = outputs.last_hidden_state.cpu().numpy()
    return desc_embeddings

  def tokenize_and_generate_embeddings_codes(self,code):
    if not self.is_loadtokenizers:
      raise ValueError("tokenizers dont load")
    # Tokenize input code and move tokens to GPU
    tokens = self.codebeart_tokenizer.encode_plus(code, return_tensors="pt", truncation=True)
    tokens = {key: value.to(device) for key, value in tokens.items()}
    # Generate embeddings
    with torch.no_grad():
        outputs = self.codebeart_model(**tokens)
    # Extract embeddings for all tokens
    code_embeddings = outputs.last_hidden_state.cpu().numpy()

    return code_embeddings


  def tokenize_and_generate_embeddings_graphcodes(self,code):
    if not self.is_loadtokenizers:
      raise ValueError("tokenizers dont load")
    # Tokenize input code and move tokens to GPU
    tokens = self.graphcodebert_tokenizer.encode_plus(code, return_tensors="pt", truncation=True)
    tokens = {key: value.to(device) for key, value in tokens.items()}
    # Generate embeddings
    with torch.no_grad():
        outputs = self.graphcodebert_model(**tokens)
    # Extract embeddings for all tokens
    code_embeddings = outputs.last_hidden_state.cpu().numpy()
    return code_embeddings

  def load_beart_tokenizer(self):
    # Verificar si la GPU está disponible
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Cargar el tokenizer y el modelo en la GPU si está disponible
    model_name = "bert-base-uncased"
    self.beart_model = BertModel.from_pretrained(model_name)
    self.beart_tokenizer = BertTokenizer.from_pretrained(model_name)
    self.beart_model.to(device)

  def load_graphcodebert_tokenizer(self):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    self.graphcodebert_tokenizer = AutoTokenizer.from_pretrained("microsoft/graphcodebert-base")
    self.graphcodebert_model = AutoModel.from_pretrained("microsoft/graphcodebert-base")
    self.graphcodebert_model.to(device)


  def load_codebert_tokenizer(self):
    # Verificar si la GPU está disponible
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Cargar el tokenizer y el modelo en la GPU si está disponible
    self.codebeart_tokenizer = RobertaTokenizer.from_pretrained("microsoft/codebert-base")
    self.codebeart_model = RobertaModel.from_pretrained("microsoft/codebert-base")
    self.codebeart_model.to(device)

  def start_tokenizer(self):
    self.load_beart_tokenizer()
    #self.load_codebert_tokenizer()
    self.load_graphcodebert_tokenizer()
    self.is_loadtokenizers=True




# All characteristics

In [25]:
def categoricallabelAll(w):

  if w=="['Initial state']":
    return 0
  if w=="['Final state']":
    return 1
  if w=="['State transformation']":
    return 2
  if w=="['Initial state', 'Final state']":
    return 3
  if w=="['Initial state', 'State transformation']":
    return 4
  if w=="['Final state', 'State transformation']":
    return 5
  if w=="['Initial state', 'Final state', 'State transformation']":
    return 6
  return 7

category=np.array([
    'Initial state',
    'Final state',
    'State transformation',
    'Initial state, Final state',
    'Initial state, State transformation',
    'Final state, State transformation',
    'Initial state, Final state, State transformation'

])

# Load Dataset

In [26]:
train_full.head()

,No.,Problema,Solución,Estado incial,Estado final,Transformación de estado,Etiqueta 1,Etiqueta 2,Realimentación
0,1,Write a Python function that returns the facto...,def factorial(n):\n result = 1\n i = 1\n...,"result = 1, i = 1",i <= n,"result *= i, i += 1",Correct,['Correct'],NaN
1,2,Write a Python function that returns the sum o...,def sum_1_to_100():\n total = 0\n i = 1\...,"total = 0, i = 1, n = 100",i <= n,"total += i, i += 1",Correct,['Correct'],NaN
2,3,Write a Python function that prints the number...,def print_and_store():\n numbers = []\n ...,"numbers = [], i = 0, n = 10",i <= n,"print(i), numbers.append(i), i += 1",Correct,['Correct'],NaN
3,4,Write a Python function to print the numbers f...,def print_and_store_reverse():\n numbers = ...,"numbers = [], i = 10, n = 1",i >= n,"print(i), numbers.append(i), i -= 1",Correct,['Correct'],NaN
4,5,Write a Python function to check if a number i...,def is_prime(num):\n if num <= 1:\n ...,"i = 2, i = = 0:",i <= num//2,"if num % i == 0:, return False, i += 1",Correct,['Correct'],NaN


In [27]:
train_full.drop(["No.","Realimentación","Estado incial","Estado final","Transformación de estado"],axis=1,inplace=True)
train_full

,Problema,Solución,Etiqueta 1,Etiqueta 2
0,Write a Python function that returns the facto...,def factorial(n):\n result = 1\n i = 1\n...,Correct,['Correct']
1,Write a Python function that returns the sum o...,def sum_1_to_100():\n total = 0\n i = 1\...,Correct,['Correct']
2,Write a Python function that prints the number...,def print_and_store():\n numbers = []\n ...,Correct,['Correct']
3,Write a Python function to print the numbers f...,def print_and_store_reverse():\n numbers = ...,Correct,['Correct']
4,Write a Python function to check if a number i...,def is_prime(num):\n if num <= 1:\n ...,Correct,['Correct']
...,...,...,...,...
4995,Write a Python function that returns the sum o...,def sum_of_first_five_numbers():\n number =...,Incorrect,"['Initial state', 'Final state']"
4996,Write a Python function that returns the facto...,def factorial(n):\n result = 0\n i = 0\n...,Incorrect,"['Initial state', 'Final state']"
4997,Write a Python function that prints the number...,def print_and_store_numbers():\n numbers = ...,Incorrect,"['Initial state', 'Final state']"
4998,Write a Python function to print the numbers f...,def print_and_store_numbers():\n numbers = ...,Incorrect,"['Initial state', 'Final state']"


In [28]:
train_full[["Estado incial","Transformación de estado","Estado final"]]=train_full['Solución'].apply(DGries_states).apply(pd.Series)

Streaming output truncated to the last 5000 lines.
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK c

In [29]:
train_full = train_full[train_full['Etiqueta 1']!='Correct'].copy()

In [30]:
y=train_full['Etiqueta 2'].apply(categoricallabelAll)
np.unique(y)

array([0, 1, 2, 3, 4, 5, 6])

In [31]:
train_full.dropna(inplace=True)

In [32]:
np.unique(train_full['Etiqueta 2'])

array(["['Final state', 'State transformation']", "['Final state']",
       "['Initial state', 'Final state', 'State transformation']",
       "['Initial state', 'Final state']",
       "['Initial state', 'State transformation']", "['Initial state']",
       "['State transformation']"], dtype=object)

In [33]:
encoder=Encoder()
encoder.start_tokenizer()

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/539 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at microsoft/graphcodebert-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [34]:
%%time
problem=train_full['Problema'].apply(encoder.tokenize_and_generate_embeddings_descriptions).to_numpy()
startstate = train_full['Estado incial'].apply(encoder.tokenize_and_generate_embeddings_graphcodes).to_numpy()
finalstate = train_full['Estado final'].apply(encoder.tokenize_and_generate_embeddings_graphcodes).to_numpy()
transstate = train_full['Transformación de estado'].apply(encoder.tokenize_and_generate_embeddings_graphcodes).to_numpy()

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

CPU times: user 2min 2s, sys: 2.23 s, total: 2min 4s
Wall time: 2min 6s


In [35]:
problem.shape,startstate.shape,finalstate.shape,transstate.shape

((3459,), (3459,), (3459,), (3459,))

In [36]:
print(startstate.shape)
print(startstate[1262].shape)


(3459,)
(1, 13, 768)


In [37]:
Xp=np.array([sentence[0].mean(axis=0) for sentence in problem])
Xs=np.array([sentence[0].mean(axis=0) for sentence in startstate])
Xf=np.array([sentence[0].mean(axis=0) for sentence in finalstate])
Xt=np.array([sentence[0].mean(axis=0) for sentence in transstate])
Xp.shape,Xs.shape,Xf.shape,Xt.shape

((3459, 768), (3459, 768), (3459, 768), (3459, 768))

In [38]:
y=train_full['Etiqueta 2'].apply(categoricallabelAll)
y=y.to_numpy()
np.unique(y)

array([0, 1, 2, 3, 4, 5, 6])

In [39]:
from sklearn.model_selection import train_test_split

Xp_train,Xp_test,Xs_train,Xs_test,Xt_train,Xt_test,Xf_train,Xf_test,y_train,y_test=train_test_split(Xp,Xs,Xt,Xf,y,test_size=0.2,random_state=2023, stratify=y)

In [40]:
Xp_train.shape,Xp_test.shape,Xs_train.shape,Xs_test.shape,Xt_train.shape,Xt_test.shape,Xf_train.shape,Xf_test.shape,y_train.shape,y_test.shape

((2767, 768),
 (692, 768),
 (2767, 768),
 (692, 768),
 (2767, 768),
 (692, 768),
 (2767, 768),
 (692, 768),
 (2767,),
 (692,))

# Keras model

In [41]:
from tensorflow.keras import backend as K
import gc
def ANNTC(input,base,pow_initial,num_max_blocks):

  drop_out=0.5
  print("base",base,"pow",pow_initial,"num_max_blocks",num_max_blocks)
  n=num_max_blocks//2
  neurons=int(base**(pow_initial+n+1))
  # Encoder
  x=None
  print("Encoder",n)
  try:
    for i in range(n):
      x=Dense(neurons)(input)
      x=BatchNormalization()(x)
      x=Activation('relu')(x)
      x=Dropout(drop_out)(x)
      input=x
      drop_out=0.2
      print("Block ",i,neurons)
      neurons=int(neurons/base)



    #BottleNeck
    x=Dense(neurons)(x)
    x=BatchNormalization()(x)
    x=Activation('relu')(x)
    x=Dropout(0.2)(x)
    print("BottleNeck",neurons)

    # Decoder
    print("Decoder",n)
    for i in range(n):
      neurons=int(neurons*base)
      print("Block ",i,neurons)
      x=Dense(neurons)(x)
      x=BatchNormalization()(x)
      x=Activation('relu')(x)
      x=Dropout(drop_out)(x)
      if i==n-1:
        drop_out=0.5
      drop_out=0.2
  except Exception as err:

    K.clear_session()
    gc.collect()
    del x
    print(f"Unexpected {err=}, {type(err)=}")
    raise

  return x


def neural_network(problem,start_input,trass_input,final_input,base=10,pow_initial=4,num_max_blocks=3):
  mlp_p=ANNTC(problem,base,pow_initial,num_max_blocks)
  mlp_s=ANNTC(start_input,base,pow_initial,num_max_blocks)
  mlp_t=ANNTC(trass_input,base,pow_initial,num_max_blocks)
  mlp_f=ANNTC(final_input,base,pow_initial,num_max_blocks)
  ## Concatenate All models
  combined = concatenate([mlp_p, mlp_s, mlp_t,mlp_f])
  ## output layer
  output=Dense(8)(combined)
  output=BatchNormalization()(output)
  output=Activation('softmax')(output)
  ## Model
  model = Model(inputs=[problem_input, start_input, trass_input,final_input], outputs=output)
  return model

In [42]:
np.logspace(2, 4, num=3, base=10)


array([  100.,  1000., 10000.])

In [43]:
dftimes=pd.read_csv("/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/ResultsGraph/times_tuning.csv",usecols=['tuning','times'])


In [44]:
dftimes['tuning']=dftimes['tuning'].apply(lambda x: x.replace(",","_"))
dftimes

,tuning,times
0,3_2_0,68.531808
1,3_2_1,71.512141
2,3_2_2,70.775239
3,3_2_3,76.419507
4,3_8_0,72.372247
5,3_8_1,65.380754
6,3_8_2,118.528536
7,3_10_0,79.211416
8,3_10_1,67.034733
9,3_10_2,413.857732


In [45]:
max_val_acuracies=[]
max_accuracies=[]
configurations=[]
# read histories csv
for num_max_blocks in [3,5,7]:
  print("Bloque")
  for base in [2,8,10,16]:
    print("")
    for pow_initial in [0,1,2,3]:
      try:
        df_histories=pd.read_csv('/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/ResultsGraph/histories_{0}.csv'.format(f"{num_max_blocks}_{base}_{pow_initial}"))
        configurations.append(f"{num_max_blocks}_{base}_{pow_initial}")
        max_val_acuracies.append(max(df_histories['val_accuracy']))
        max_accuracies.append(max(df_histories['accuracy']))
        print(f"{num_max_blocks}_{base}_{pow_initial}")
        print("max_val_accuracy",max(df_histories['val_accuracy']))
        print("max_accuracy",max(df_histories['accuracy']))
      except:
        print("Error")

Bloque

3_2_0
max_val_accuracy 0.7310469150543213
max_accuracy 0.6249434947967529
3_2_1
max_val_accuracy 0.8411552309989929
max_accuracy 0.7799367308616638
3_2_2
max_val_accuracy 0.8754512667655945
max_accuracy 0.8721193075180054
3_2_3
max_val_accuracy 0.911552369594574
max_accuracy 0.9367374777793884

3_8_0
max_val_accuracy 0.9043321013450624
max_accuracy 0.9380930662155152
3_8_1
max_val_accuracy 0.9368231296539308
max_accuracy 0.9959331154823304
3_8_2
max_val_accuracy 0.929602861404419
max_accuracy 1.0
Error

3_10_0
max_val_accuracy 0.9314079284667968
max_accuracy 0.9602349996566772
3_10_1
max_val_accuracy 0.9386281371116638
max_accuracy 0.9981924891471864
3_10_2
max_val_accuracy 0.9332129955291748
max_accuracy 1.0
Error

3_16_0
max_val_accuracy 0.9314079284667968
max_accuracy 0.9805693626403807
3_16_1
max_val_accuracy 0.9314079284667968
max_accuracy 1.0
Error
Error
Bloque

5_2_0
max_val_accuracy 0.7057761549949646
max_accuracy 0.6597378849983215
5_2_1
max_val_accuracy 0.824909746646

In [46]:
dfall=pd.DataFrame({"configurations":configurations,"max_val_acuracies":max_val_acuracies,"max_accuracies":max_accuracies})
dfall

,configurations,max_val_acuracies,max_accuracies
0,3_2_0,0.731047,0.624943
1,3_2_1,0.841155,0.779937
2,3_2_2,0.875451,0.872119
3,3_2_3,0.911552,0.936737
4,3_8_0,0.904332,0.938093
5,3_8_1,0.936823,0.995933
6,3_8_2,0.929603,1.000000
7,3_10_0,0.931408,0.960235
8,3_10_1,0.938628,0.998192
9,3_10_2,0.933213,1.000000


In [47]:
df_merged = pd.merge(dfall, dftimes, left_on='configurations', right_on='tuning')
df_merged

,configurations,max_val_acuracies,max_accuracies,tuning,times
0,3_2_0,0.731047,0.624943,3_2_0,68.531808
1,3_2_1,0.841155,0.779937,3_2_1,71.512141
2,3_2_2,0.875451,0.872119,3_2_2,70.775239
3,3_2_3,0.911552,0.936737,3_2_3,76.419507
4,3_8_0,0.904332,0.938093,3_8_0,72.372247
5,3_8_1,0.936823,0.995933,3_8_1,65.380754
6,3_8_2,0.929603,1.000000,3_8_2,118.528536
7,3_10_0,0.931408,0.960235,3_10_0,79.211416
8,3_10_1,0.938628,0.998192,3_10_1,67.034733
9,3_10_2,0.933213,1.000000,3_10_2,413.857732


In [48]:
from sklearn.metrics import matthews_corrcoef, precision_recall_curve, auc
from sklearn.preprocessing import label_binarize
import numpy as np

def calculate_mcc_multiclass(y_true, y_pred_probs):
    y_pred_labels = np.argmax(y_pred_probs, axis=1)
    # Convertir etiquetas verdaderas one-hot a etiquetas enteras si es necesario
    #y_true_labels = np.argmax(y_true, axis=1) if y_true.ndim > 1 else y_true
    return matthews_corrcoef(y_true, y_pred_labels)

def calculate_auc_pr_multiclass(y_true, y_pred_probs, average='macro'):
    n_classes = y_pred_probs.shape[1]
    #y_true_labels = np.argmax(y_true, axis=1) if y_true.ndim > 1 else y_true
    y_true_bin = label_binarize(y_true, classes=range(n_classes))

    auc_pr_list = []
    for i in range(n_classes):
        precision, recall, _ = precision_recall_curve(y_true_bin[:, i], y_pred_probs[:, i])
        auc_pr_list.append(auc(recall, precision))

    if average == 'macro':
        return np.mean(auc_pr_list)
    elif average == 'weighted':
        class_counts = y_true_bin.sum(axis=0)
        return np.average(auc_pr_list, weights=class_counts)
    else:
        return auc_pr_list


In [49]:
from time import time
from tensorflow import keras
times_predict=[]
accuracies_predict=[]
loss_predict=[]
mccs_predict=[]
aucpr_predict=[]
for num_max_blocks in [3,5,7]:
  print("Bloque")
  for base in [2,8,10,16]:
    print("")
    for pow_initial in [0,1,2,3]:
      try:
        model=keras.models.load_model('/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/ResultsGraph/best_finall_{0}.keras'.format(f"{num_max_blocks}_{base}_{pow_initial}"))

        evaluate=model.evaluate([Xp_test,Xs_test,Xt_test,Xf_test],y_test)


        t1=time()
        y_pred=model.predict([Xp_test,Xs_test,Xt_test,Xf_test])
        t2=time()
        mcc=calculate_mcc_multiclass(y_test, y_pred)
        auc_pr=calculate_auc_pr_multiclass(y_test, y_pred)

        times_predict.append(t2-t1)
        accuracies_predict.append(evaluate[1])
        loss_predict.append(evaluate[0])
        mccs_predict.append(mcc)
        aucpr_predict.append(auc_pr)


        accuracy=evaluate[1]
        loss=evaluate[0]

        print(f"{num_max_blocks}_{base}_{pow_initial}")
        print("accuracy",accuracy)
        print("loss",loss)
        print("mcc",mcc)
        print("auc_pr",auc_pr)
        print("time predict",t2-t1)
      except:
        print("Error")


Bloque

22/22 ━━━━━━━━━━━━━━━━━━━━ 3s 48ms/step - accuracy: 0.7416 - loss: 0.9536
22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 39ms/step
3_2_0
accuracy 0.7283236980438232
loss 0.9844395518302917
mcc 0.6316752250654477
auc_pr 0.5454157205862538
time predict 2.0273234844207764


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


22/22 ━━━━━━━━━━━━━━━━━━━━ 3s 34ms/step - accuracy: 0.8292 - loss: 0.6223
22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 40ms/step
3_2_1
accuracy 0.8135837912559509
loss 0.6526886820793152
mcc 0.7474694121402162
auc_pr 0.7163971689485615
time predict 1.675412893295288


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


22/22 ━━━━━━━━━━━━━━━━━━━━ 3s 50ms/step - accuracy: 0.8811 - loss: 0.4660
22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 40ms/step
3_2_2
accuracy 0.8699421882629395
loss 0.5103393793106079
mcc 0.8250069270673737
auc_pr 0.7826841932833178
time predict 1.9014289379119873


...

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


22/22 ━━━━━━━━━━━━━━━━━━━━ 3s 47ms/step - accuracy: 0.8895 - loss: 0.3046
22/22 ━━━━━━━━━━━━━━━━━━━━ 3s 76ms/step
7_8_0
accuracy 0.8901734352111816
loss 0.34000450372695923
mcc 0.8531635902105588
auc_pr 0.8665253332096574
time predict 3.201630115509033
Error
Error
Error



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


22/22 ━━━━━━━━━━━━━━━━━━━━ 5s 51ms/step - accuracy: 0.8979 - loss: 0.3504
22/22 ━━━━━━━━━━━━━━━━━━━━ 3s 85ms/step
7_10_0
accuracy 0.8959537744522095
loss 0.3806202709674835
mcc 0.8607190126932434
auc_pr 0.8629657188104993
time predict 3.458306312561035
Error
Error
Error

Error
Error
Error
Error


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


In [50]:
df_merged['predict_time']=times_predict
df_merged['predict_accuracy']=accuracies_predict
df_merged['predict_loss']=loss_predict
df_merged

,configurations,max_val_acuracies,max_accuracies,tuning,times,predict_time,predict_accuracy,predict_loss
0,3_2_0,0.731047,0.624943,3_2_0,68.531808,2.027323,0.728324,0.984440
1,3_2_1,0.841155,0.779937,3_2_1,71.512141,1.675413,0.813584,0.652689
2,3_2_2,0.875451,0.872119,3_2_2,70.775239,1.901429,0.869942,0.510339
3,3_2_3,0.911552,0.936737,3_2_3,76.419507,1.626259,0.894509,0.358785
4,3_8_0,0.904332,0.938093,3_8_0,72.372247,1.817180,0.891618,0.398516
5,3_8_1,0.936823,0.995933,3_8_1,65.380754,1.643635,0.920520,0.294379
6,3_8_2,0.929603,1.000000,3_8_2,118.528536,1.718147,0.914740,0.325460
7,3_10_0,0.931408,0.960235,3_10_0,79.211416,1.646883,0.906069,0.341141
8,3_10_1,0.938628,0.998192,3_10_1,67.034733,1.663238,0.921965,0.288719
9,3_10_2,0.933213,1.000000,3_10_2,413.857732,2.575073,0.923410,0.321796


In [51]:
df_merged['predict_mcc']=mccs_predict
df_merged['predict_aucpr']=aucpr_predict

In [52]:
df_merged

,configurations,max_val_acuracies,max_accuracies,tuning,times,predict_time,predict_accuracy,predict_loss,predict_mcc,predict_aucpr
0,3_2_0,0.731047,0.624943,3_2_0,68.531808,2.027323,0.728324,0.984440,0.631675,0.545416
1,3_2_1,0.841155,0.779937,3_2_1,71.512141,1.675413,0.813584,0.652689,0.747469,0.716397
2,3_2_2,0.875451,0.872119,3_2_2,70.775239,1.901429,0.869942,0.510339,0.825007,0.782684
3,3_2_3,0.911552,0.936737,3_2_3,76.419507,1.626259,0.894509,0.358785,0.858472,0.852143
4,3_8_0,0.904332,0.938093,3_8_0,72.372247,1.817180,0.891618,0.398516,0.854767,0.849284
5,3_8_1,0.936823,0.995933,3_8_1,65.380754,1.643635,0.920520,0.294379,0.893812,0.891678
6,3_8_2,0.929603,1.000000,3_8_2,118.528536,1.718147,0.914740,0.325460,0.886292,0.878756
7,3_10_0,0.931408,0.960235,3_10_0,79.211416,1.646883,0.906069,0.341141,0.874150,0.877399
8,3_10_1,0.938628,0.998192,3_10_1,67.034733,1.663238,0.921965,0.288719,0.895738,0.890661
9,3_10_2,0.933213,1.000000,3_10_2,413.857732,2.575073,0.923410,0.321796,0.897862,0.877122


In [53]:
df_merged.to_csv("/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/ResultsGraph/fine_tuning.csv",index=False)

In [54]:
df_merged[df_merged['predict_aucpr']>0.88]

,configurations,max_val_acuracies,max_accuracies,tuning,times,predict_time,predict_accuracy,predict_loss,predict_mcc,predict_aucpr
5,3_8_1,0.936823,0.995933,3_8_1,65.380754,1.643635,0.920520,0.294379,0.893812,0.891678
8,3_10_1,0.938628,0.998192,3_10_1,67.034733,1.663238,0.921965,0.288719,0.895738,0.890661
11,3_16_1,0.931408,1.000000,3_16_1,121.192008,1.990958,0.921965,0.319015,0.895908,0.881822
17,5_8_1,0.927798,0.999548,5_8_1,147.187276,3.250433,0.919075,0.317843,0.892113,0.882208


In [56]:
df_merged[(df_merged['predict_aucpr']>0.88)&
          (df_merged['max_accuracies']-df_merged['max_val_acuracies']<0.06)]

,configurations,max_val_acuracies,max_accuracies,tuning,times,predict_time,predict_accuracy,predict_loss,predict_mcc,predict_aucpr
5,3_8_1,0.936823,0.995933,3_8_1,65.380754,1.643635,0.920520,0.294379,0.893812,0.891678
8,3_10_1,0.938628,0.998192,3_10_1,67.034733,1.663238,0.921965,0.288719,0.895738,0.890661
